# 15.1 Minimum-cost transshipment models

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

As a concrete example, imagine that the nodes and arcs in [Figure 15-1](./15_1_minimumcost_transshipment_models.ipynb#fig-15-1) represent cities
and intercity transportation links. A manufacturing plant at the city marked PITT will

```ampl
BOS 90
                                                           1.7, 100

                                                   NE            0.7, 100            EWR 120

                       2.5, 250
                                                           1.3, 100
     450 PITT                                                                         BWI 120
                                                   1.3, 100
                       3.5, 250                                          0.8, 100
                                                   SE                                 ATL 70
                                                                 0.2, 100
                                                           2.1, 100
                                                                                     MCO 50
```

**[Figure 15-1](./15_1_minimumcost_transshipment_models.ipynb#fig-15-1):** A directed network.

make 450,000 packages of a certain product in the next week, as indicated by the 450 at
the left of the diagram. The cities marked NE and SE are the northeast and southeast distribution
centers, which receive packages from the plant and transship them to warehouses
at the cities coded as BOS, EWR, BWI, ATL and MCO. (Frequent flyers will recognize
Boston, Newark, Baltimore, Atlanta, and Orlando.) These warehouses require 90, 120, 120, 70
and 50 thousand packages, respectively, as indicated by the numbers at the
right. For each intercity link there is a shipping cost per thousand packages and an upper
limit on the packages that can be shipped, indicated by the two numbers next to the corresponding
arrow in the diagram.

The optimization problem over this network is to find the lowest-cost plan of shipments
that uses only the available links, respects the specified capacities, and meets the
requirements at the warehouses. We first model this as a general network flow problem,
and then consider alternatives that specialize the model to the particular situation at hand.
We conclude by introducing a few of the most common variations on the network flow
constraints.

**A general transshipment model**

To write a model for any problem of shipments from city to city, we can start by
defining a set of cities and a set of links. Each link is in turn defined by a start city and an
end city, so we want the set of links to be a subset of the set of ordered pairs of cities:

```ampl
set CITIES;
set LINKS within (CITIES cross CITIES);
```

Corresponding to each city there is potentially a supply of packages and a demand for
packages:

```ampl
param supply {CITIES} >= 0;
param demand {CITIES} >= 0;
```

In the case of the problem described by [Figure 15-1](./15_1_minimumcost_transshipment_models.ipynb#fig-15-1), the only nonzero value of supply
should be the one for PITT, where packages are manufactured and supplied to the distribution
network. The only nonzero values of demand should be those corresponding to
the five warehouses.

The costs and capacities are indexed over the links:

```ampl
param cost {LINKS} >= 0;
param capacity {LINKS} >= 0;
```

as are the decision variables, which represent the amounts to ship over the links. These
variables are nonnegative and bounded by the capacities:

```ampl
var Ship {(i,j) in LINKS} >= 0, <= capacity[i,j];
```

The objective is

```ampl
minimize Total_Cost:
       sum {(i,j) in LINKS} cost[i,j] * Ship[i,j];
```

which represents the sum of the shipping costs over all of the links.

It remains to describe the constraints. At each city, the packages supplied plus packages
shipped in must balance the packages demanded plus packages shipped out:

```ampl
subject to Balance {k in CITIES}:
       supply[k] + sum {(i,k) in LINKS} Ship[i,k]
       = demand[k] + sum {(k,j) in LINKS} Ship[k,j];
```

Because the expression

```ampl
sum {(i,k) in LINKS} Ship[i,k]
```

appears within the scope of definition of the dummy index k, the summation is interpreted
to run over all cities `i` such that (i,k) is in LINKS. That is, the summation is
over all links into city k; similarly, the second summation is over all links out of k. This
indexing convention, which was explained in [Section 6.2](../06/6_2_subsets_and_slices_of_ordered_pairs.ipynb), is frequently useful in describing
network balance constraints algebraically. Figures 15-2a and 15-2b display the complete
model and data for the particular problem depicted in [Figure 15-1](./15_1_minimumcost_transshipment_models.ipynb#fig-15-1).

If all of the variables are moved to the left of the = sign and the constants to the right,
the Balance constraint becomes:

```ampl
subject to Balance {k in CITIES}:
       sum {(i,k) in LINKS} Ship[i,k]
       - sum {(k,j) in LINKS} Ship[k,j]
       = demand[k] - supply[k];
```

This variation may be interpreted as saying that, at each city k, shipments in minus shipments
out must equal "net demand". If no city has both a plant and a warehouse (as in

```ampl
set CITIES;
set LINKS within (CITIES cross CITIES);
param supply {CITIES} >= 0;             # amounts available at cities
param demand {CITIES} >= 0;             # amounts required at cities
check: sum {i in CITIES} supply[i] = sum {j in CITIES} demand[j];
param cost {LINKS} >= 0;                # shipment costs/1000 packages
param capacity {LINKS} >= 0;            # max packages that can be shipped
var Ship {(i,j) in LINKS} >= 0, <= capacity[i,j];
                                   # packages to be shipped
minimize Total_Cost:
       sum {(i,j) in LINKS} cost[i,j] * Ship[i,j];
subject to Balance {k in CITIES}:
       supply[k] + sum {(i,k) in LINKS} Ship[i,k]
              = demand[k] + sum {(k,j) in LINKS} Ship[k,j];
```

**[Figure 15-2a](./15_1_minimumcost_transshipment_models.ipynb#fig-15-2a):** General transshipment model (net1.mod).

```ampl
set CITIES := PITT  NE SE  BOS EWR BWI ATL MCO ;
set LINKS := (PITT,NE) (PITT,SE)
              (NE,BOS) (NE,EWR) (NE,BWI)
              (SE,EWR) (SE,BWI) (SE,ATL) (SE,MCO);
param supply                          default 0 := PITT 450 ;
param demand default 0 :=
BOS 90, EWR 120, BWI 120,  ATL          70,           MCO          50;
param:          cost   capacity  :=
       PITT NE   2.5     250
       PITT SE   3.5     250
       NE BOS    1.7     100
       NE EWR    0.7     100
       NE BWI    1.3     100
       SE EWR    1.3     100
       SE BWI    0.8     100
       SE ATL    0.2     100
       SE MCO    2.1     100 ;
```

**[Figure 15-2b](./15_1_minimumcost_transshipment_models.ipynb#fig-15-2b):** Data for general transshipment model (net1.dat).

our example), then positive net demand always indicates warehouse cities, negative net
demand indicates plant cities, and zero net demand indicates transshipment cities. Thus
we could have gotten by with just one parameter net_demand in place of demand and
supply, with the sign of net_demand[k] indicating what goes on at city k. Alternative
formulations of this kind are often found in descriptions of network flow models.
**Specialized transshipment models**

The preceding general approach has the advantage of being able to accommodate any
pattern of supplies, demands, and links between cities. For example, a simple change in
the data would suffice to model a plant at one of the distribution centers, or to allow shipment
links between some of the warehouses.

The disadvantage of a general formulation is that it fails to show clearly what arrangement
of supplies, demands and links is expected, and in fact will allow inappropriate
arrangements. If we know that the situation will be like the one shown in [Figure 15-1](./15_1_minimumcost_transshipment_models.ipynb#fig-15-1),
with supply at one plant, which ships to distribution centers, which then ship to warehouses
that satisfy demand, the model can be specialized to exhibit and enforce such a
structure.

To show explicitly that there are three different kinds of cities in the specialized
model, we can declare them separately. We use a symbolic parameter rather than a set to
hold the name of the plant, to specify that only one plant is expected:

```ampl
param p_city symbolic;
set D_CITY;
set W_CITY;
```

There must be a link between the plant and each distribution center, so we need a subset
of pairs only to specify which links connect distribution centers to warehouses:

```ampl
set DW_LINKS within (D_CITY cross W_CITY);
```

With the declarations organized in this way, it is impossible to specify inappropriate
kinds of links, such as ones between two warehouses or from a warehouse back to the
plant.

One parameter represents the supply at the plant, and a collection of demand parameters
is indexed over the warehouses:

```ampl
param p_supply >= 0;
param w_demand {W_CITY} >= 0;
```

These declarations allow supply and demand to be defined only where they belong.

At this juncture, we can define the sets CITIES and LINKS and the parameters
supply and demand as they would be required by our previous model:

```ampl
set CITIES = {p_city} union D_CITY union W_CITY;
set LINKS = ({p_city} cross D_CITY) union DW_LINKS;
param supply {k in CITIES} =
       if k = p_city then p_supply else 0;
param demand {k in CITIES} =
       if k in W_CITY then w_demand[k] else 0;
```

The rest of the model can then be exactly as in the general case, as indicated in
Figures 15-3a and 15-3b.

```ampl
param p_city symbolic;
set D_CITY;
set W_CITY;
set DW_LINKS within (D_CITY cross W_CITY);
param p_supply >= 0;                    # amount available at plant
param w_demand {W_CITY} >= 0;           # amounts required at warehouses
              check: p_supply = sum {k in W_CITY} w_demand[k];
set CITIES = {p_city} union D_CITY union W_CITY;
set LINKS = ({p_city} cross D_CITY) union DW_LINKS;
param supply {k in CITIES} =
       if k = p_city then p_supply else 0;
param demand {k in CITIES} =
       if k in W_CITY then w_demand[k] else 0;
### Remainder same as general transshipment model ###
param cost {LINKS} >= 0;                # shipment costs/1000 packages
param capacity {LINKS} >= 0;            # max packages that can be shipped
var Ship {(i,j) in LINKS} >= 0, <= capacity[i,j];
                                   # packages to be shipped
minimize Total_Cost:
       sum {(i,j) in LINKS} cost[i,j] * Ship[i,j];
subject to Balance {k in CITIES}:
       supply[k] + sum {(i,k) in LINKS} Ship[i,k]
       = demand[k] + sum {(k,j) in LINKS} Ship[k,j];
```

**[Figure 15-3a](./15_1_minimumcost_transshipment_models.ipynb#fig-15-3a):** Specialized transshipment model (net2.mod).

Alternatively, we can maintain references to the different types of cities and links
throughout the model. This means that we must declare two types of costs, capacities and
shipments:

```ampl
param pd_cost {D_CITY} >= 0;
param dw_cost {DW_LINKS} >= 0;
param pd_cap {D_CITY} >= 0;
param dw_cap {DW_LINKS} >= 0;
var PD_Ship {i in D_CITY} >= 0, <= pd_cap[i];
var DW_Ship {(i,j) in DW_LINKS} >= 0, <= dw_cap[i,j];
```

The "pd" quantities are associated with shipments from the plant to distribution centers;
because they all relate to shipments from the same plant, they need only be indexed over
D_CITY. The "dw" quantities are associated with shipments from distribution centers
to warehouses, and so are naturally indexed over DW_LINKS.

The total shipment cost can now be given as the sum of two summations:

```ampl
param p_city := PITT ;
set D_CITY := NE SE ;
set W_CITY := BOS EWR BWI ATL MCO ;
set DW_LINKS := (NE,BOS) (NE,EWR) (NE,BWI)
                     (SE,EWR) (SE,BWI) (SE,ATL) (SE,MCO);
param p_supply := 450 ;
param w_demand :=
       BOS 90, EWR 120, BWI 120,  ATL 70, MCO  50;
param:           cost  capacity :=
       PITT NE   2.5    250
       PITT SE   3.5    250
       NE BOS    1.7    100
       NE EWR    0.7    100
       NE BWI    1.3    100
       SE EWR    1.3    100
       SE BWI    0.8    100
       SE ATL    0.2    100
       SE MCO    2.1    100 ;
```

**[Figure 15-3b](./15_1_minimumcost_transshipment_models.ipynb#fig-15-3b):** Data for specialized transshipment model (net2.dat).

```ampl
minimize Total_Cost:
       sum {i in D_CITY} pd_cost[i] * PD_Ship[i]
       + sum {(i,j) in DW_LINKS} dw_cost[i,j] * DW_Ship[i,j];
```

Finally, there must be three kinds of balance constraints, one for each kind of city. Shipments
from the plant to the distribution centers must equal the supply at the plant:

```ampl
subject to P_Bal: sum {i in D_CITY} PD_Ship[i] = p_supply;
```

At each distribution center, shipments in from the plant must equal shipments out to all
the warehouses:

```ampl
subject to D_Bal {i in D_CITY}:
       PD_Ship[i] = sum {(i,j) in DW_LINKS} DW_Ship[i,j];
```

And at each warehouse, shipments in from all distribution centers must equal the
demand:

```ampl
subject to W_Bal {j in W_CITY}:
       sum {(i,j) in DW_LINKS} DW_Ship[i,j] = w_demand[j];
```

The whole model, with appropriate data, is shown in Figures 15-4a and 15-4b.

The approaches shown in Figures 15-3 and 15-4 are equivalent, in the sense that they
cause the same linear program to be solved. The former is more convenient for experimenting
with different network structures, since any changes affect only the data for the
initial declarations in the model. If the network structure is unlikely to change, however,

```ampl
set D_CITY;
set W_CITY;
set DW_LINKS within (D_CITY cross W_CITY);
param p_supply >= 0;                  # amount available at plant
param w_demand {W_CITY} >= 0;         # amounts required at warehouses
       check: p_supply = sum {j in W_CITY} w_demand[j];
param pd_cost {D_CITY} >= 0;           # shipment costs/1000 packages
param dw_cost {DW_LINKS} >= 0;
param pd_cap {D_CITY} >= 0;            # max packages that can be shipped
param dw_cap {DW_LINKS} >= 0;
var PD_Ship {i in D_CITY} >= 0, <= pd_cap[i];
var DW_Ship {(i,j) in DW_LINKS} >= 0, <= dw_cap[i,j];
                            # packages to be shipped
minimize Total_Cost:
sum {i in D_CITY} pd_cost[i] * PD_Ship[i] +
sum {(i,j) in DW_LINKS} dw_cost[i,j] * DW_Ship[i,j];
subject to P_Bal: sum {i in D_CITY} PD_Ship[i] = p_supply;
subject to D_Bal {i in D_CITY}:
PD_Ship[i] = sum {(i,j) in DW_LINKS} DW_Ship[i,j];
subject to W_Bal {j in W_CITY}:
sum {(i,j) in DW_LINKS} DW_Ship[i,j] = w_demand[j];
```

**[Figure 15-4a](./15_1_minimumcost_transshipment_models.ipynb#fig-15-4a):** Specialized transshipment model, version 2 (net3.mod).

the latter form facilitates alterations that affect only particular kinds of cities, such as the
generalizations we describe next.

**Variations on transshipment models**

Some balance constraints in a network flow model may have to be inequalities rather
than equations. In the example of Figure 15-4, if production at the plant can sometimes
exceed total demand at the warehouses, we should replace = by <= in the P_Bal constraints.

A more substantial modification occurs when the quantity of flow that comes out of
an `arc` does not necessarily equal the quantity that went in. As an example, a small fraction
of the packages shipped from the plant may be damaged or stolen before the packages
reach the distribution center. Suppose that a parameter pd_loss is introduced to
represent the loss rate:

```ampl
param pd_loss {D_CITY} >= 0, < 1;
```

Then the balance constraints at the distribution centers must be adjusted accordingly:

```ampl
set D_CITY := NE SE ;
set W_CITY := BOS EWR BWI ATL MCO ;
set DW_LINKS := (NE,BOS) (NE,EWR) (NE,BWI)
                     (SE,EWR) (SE,BWI) (SE,ATL) (SE,MCO);
param p_supply := 450 ;
param w_demand :=
       BOS 90, EWR 120,  BWI 120,  ATL  70,  MCO 50;
param:      pd_cost   pd_cap :=
       NE       2.5     250
       SE       3.5     250 ;
param:        dw_cost    dw_cap :=
       NE BOS     1.7      100
       NE EWR     0.7      100
       NE BWI     1.3      100
       SE EWR     1.3      100
       SE BWI     0.8      100
       SE ATL     0.2      100
       SE MCO     2.1      100 ;
```

**[Figure 15-4b](./15_1_minimumcost_transshipment_models.ipynb#fig-15-4b):** Data for specialized transshipment model, version 2 (net3.dat).

```ampl
subject to D_Bal {i in D_CITY}:
       (1-pd_loss[i]) * PD_Ship[i]
       = sum {(i,j) in DW_LINKS} DW_Ship[i,j];
```

The expression to the left of the = sign has been modified to reflect the fact that only
(1-pd_loss[i]) * PD_Ship[i] packages arrive at city `i` when PD_Ship[i]
packages are shipped from the plant.

A similar variation occurs when the flow is not measured in the same units throughout
the network. If demand is reported in cartons rather than thousands of packages, for
example, the model will require a parameter to represent packages per carton:

```ampl
param ppc integer > 0;
```

Then the demand constraints at the warehouses are adjusted as follows:

```ampl
subject to W_Bal {j in W_CITY}:
       sum {(i,j) in DW_LINKS} (1000/ppc) * DW_Ship[i,j]
       = w_demand[j];
```

The term (1000/ppc) * DW_Ship[i,j] represents the number of cartons received
at warehouse `j` when DW_Ship[i,j] thousand packages are shipped from distribution
center i.

```ampl
50
           a                           b
                                                   20
     100                          40
                        60                         50
           c                           d                         e
                                           60                        70
                        20
                                                   70
                                       f                         g
```

**[Figure 15-5](./15_2_other_network_models.ipynb#fig-15-5):** Traffic flow network.